# 00 · Setup & Environment Verification

> **OWNER:** BOTH MEMBERS — run this before anything else, on every machine/Colab
> account you train from.
> **PREREQUISITES:** none. This *is* the prerequisite for every other notebook.
> **EXPECTED RUNTIME:** ~10 minutes (mostly the `pip install -e ".[ml]"` and the
> COCO weights download).
> **OUTPUTS:** a printed green/red checklist, and a recommended `batch=` size for
> whatever GPU (or lack of one) you have. Nothing is written to disk beyond the
> downloaded `yolo11s.pt` COCO weights (gitignored, harmless to re-download).

If this notebook is not all-green, stop — every later notebook assumes a working
`ultralytics` + `torch` stack and will fail in more confusing ways if this one
would have failed first.

In [ ]:
# Cell 1/3 — minimal bootstrap (Colab vs local). No repo imports yet: on a
# fresh Colab runtime nothing has been cloned, so this cell is deliberately
# self-contained and only prepares sys.path so `common/` becomes importable.
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()
REPO_URL = "https://github.com/ad8thya/SmartIndiaHackathon.git"

if IN_COLAB:
    REPO_ROOT = Path("/content/SmartIndiaHackathon")
    if not REPO_ROOT.exists():
        print(f"cloning {REPO_URL} -> {REPO_ROOT}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f"{REPO_ROOT} already present locally on this runtime")
else:
    _here = Path.cwd().resolve()
    _candidates = [c for c in (_here, *_here.parents) if (c / "pyproject.toml").exists() and (c / "notebooks").exists()]
    if not _candidates:
        raise RuntimeError(
            "Could not find the repo root (looked for pyproject.toml + notebooks/ "
            f"walking up from {_here}). Run this notebook from inside the repo checkout."
        )
    REPO_ROOT = _candidates[0]

for _p in (str(REPO_ROOT), str(REPO_ROOT / "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Colab: {IN_COLAB}")
print(f"repo root: {REPO_ROOT}")

In [ ]:
# Cell 2/3 — install the ML extras. Quiet; ~60-90s on a fresh Colab runtime,
# near-instant if already installed (pip no-ops on a satisfied requirement).
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[ml]"],
    check=True,
)
print("ml extras installed")

In [ ]:
# Cell 3/3 — full environment setup (Drive mount + DATA_ROOT/MODEL_ROOT) and
# an import check: torch version + CUDA availability, so a broken environment
# fails here, not forty minutes into a training run.
from common import colab as colab_mod

env = colab_mod.setup_environment()
REPO_ROOT, DATA_ROOT, MODEL_ROOT = env["repo_root"], env["data_root"], env["model_root"]

import ultralytics

print(f"ultralytics {ultralytics.__version__}")
gpu_info = colab_mod.gpu_report()

## Disk space check

In [ ]:
import shutil

_usage = shutil.disk_usage(REPO_ROOT)
free_gb = _usage.free / (1024**3)
print(f"free disk space at {REPO_ROOT}: {free_gb:.1f} GB")
DISK_OK = free_gb >= 30.0
if not DISK_OK:
    print(
        "WARNING: under 30 GB free. RDD2022's India subset, the hazard/plate "
        "datasets, YOLO checkpoints and export artifacts can add up to more "
        "than this. Free up space or move to a machine/Colab runtime with more."
    )

## COCO weights smoke test

Download `yolo11s.pt` and run one inference to prove the stack actually works end to end, not just that the imports succeed.

In [ ]:
from ultralytics import YOLO
from ultralytics.utils import ASSETS

_smoke_model = YOLO("yolo11s.pt")  # auto-downloads COCO weights on first use
_smoke_result = _smoke_model.predict(source=str(ASSETS / "bus.jpg"), verbose=False)[0]

INFERENCE_OK = len(_smoke_result.boxes) > 0
print(f"inference smoke test: {len(_smoke_result.boxes)} detections on the sample image")
if not INFERENCE_OK:
    print("WARNING: zero detections on a known-good sample image — something is wrong with the model/weights.")

## Checklist + recommended batch size

In [ ]:
checks = [
    ("ultralytics import", True),
    ("torch import", True),
    ("CUDA available", gpu_info["cuda_available"]),
    (f"disk space >= 30 GB ({free_gb:.1f} GB free)", DISK_OK),
    ("COCO weights downloaded + inference smoke test", INFERENCE_OK),
]

print()
print("=" * 60)
for label, passed in checks:
    icon = "\u2705" if passed else "\u26a0\ufe0f "
    print(f"  {icon} {label}")
print("=" * 60)

if not gpu_info["cuda_available"]:
    print()
    print("NO GPU DETECTED on this runtime.")
    print("Local CPU training will work for this notebook's smoke test only —")
    print("for 01-06 (real training runs) switch to Colab with a GPU runtime:")
    print("  https://colab.research.google.com  ->  Runtime > Change runtime type > T4 GPU")

batch = colab_mod.recommended_batch_size(gpu_info.get("vram_gb"))
print()
print(f"recommended starting batch size at imgsz=640: batch={batch}")
print("(if you hit a CUDA OOM during training, halve it and retry rather than trusting this blindly)")

all_green = all(passed for _, passed in checks)
print()
print("ALL GREEN — proceed to 01/03." if all_green else "NOT all green — fix the warnings above before continuing.")

---
### What this notebook produced
- A verified `ultralytics` + `torch` install, with GPU/CPU confirmed.
- `yolo11s.pt` COCO weights cached locally (auto re-used by every training notebook).
- A recommended `batch=` size for your hardware.

### Next
- **Member A (M1):** `01_prepare_rdd2022.ipynb`
- **Member B (M4):** `03_prepare_hazards.ipynb` — start this one first, it is the
  slowest human task on the whole ML track and does not need a GPU while you wait.